# Volatility Regime Analysis

## Objective

Volatility is one of the most important drivers of intraday market behavior.

This notebook investigates how trading signals perform under different volatility environments.

## Research Questions

- Does signal effectiveness depend on volatility?
- Are high-volatility periods more predictable?
- Which volatility regime offers the strongest opportunities?

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("./NIFTY_50_minute.csv")

df['date'] = pd.to_datetime(
    df['date'],
    format='%d-%m-%Y %H:%M'
)

df = df.sort_values('date')
df = df.set_index('date')

df.head()

,open,high,low,close,volume
date,,,,,
2015-01-09 09:15:00,8285.45,8295.90,8285.45,8292.10,0
2015-01-09 09:16:00,8292.60,8293.60,8287.20,8288.15,0
2015-01-09 09:17:00,8287.40,8293.90,8287.40,8293.90,0
2015-01-09 09:18:00,8294.25,8300.65,8293.90,8300.65,0
2015-01-09 09:19:00,8300.60,8301.30,8298.75,8301.20,0


In [4]:
# ============================================================
# DATA CLEANING
# ============================================================

market_open = pd.Timestamp("09:15").time()
market_close = pd.Timestamp("15:29").time()

df = df[
    (df.index.time >= market_open) &
    (df.index.time <= market_close)
].copy()

df = df[
    ~df.index.duplicated(keep="first")
]

print("Rows:", len(df))
print("Start:", df.index.min())
print("End:", df.index.max())

Rows: 974705
Start: 2015-01-09 09:15:00
End: 2025-07-25 15:29:00


In [5]:
daily_close = (
    df.between_time("15:15","15:15")
      ['close']
      .copy()
)

daily_returns = daily_close.pct_change()

vol_df = pd.DataFrame({
    'date': daily_close.index.date,
    'daily_return': daily_returns.values
})

vol_df['vol20'] = (
    vol_df['daily_return']
    .rolling(20)
    .std()
    .shift(1)
)

vol_df['date'] = pd.to_datetime(
    vol_df['date']
)

vol_df.head()

,date,daily_return,vol20
0,2015-01-09,NaN,NaN
1,2015-01-12,0.004242,NaN
2,2015-01-13,-0.002091,NaN
3,2015-01-14,-0.003974,NaN
4,2015-01-15,0.028784,NaN


In [6]:

research = []

for day, day_df in df.groupby(df.index.date):

    open_bar = day_df.between_time(
        "09:15",
        "09:15"
    )

    eleven_bar = day_df.between_time(
        "11:00",
        "11:00"
    )

    close_bar = day_df.between_time(
        "15:15",
        "15:15"
    )

    if (
        len(open_bar) == 0
        or len(eleven_bar) == 0
        or len(close_bar) == 0
    ):
        continue

    day_open = open_bar.iloc[0]["open"]

    eleven_close = eleven_bar.iloc[0]["close"]

    day_close = close_bar.iloc[0]["close"]

    morning_return = (
        eleven_close / day_open - 1
    )

    research.append({
        "date": pd.Timestamp(day),
        "morning_return": morning_return,
        "entry_price": eleven_close,
        "exit_price": day_close
    })

research = pd.DataFrame(research)

research.head()

,date,morning_return,entry_price,exit_price
0,2015-01-09,0.000278,8287.75,8285.45
1,2015-01-12,-0.000476,8287.40,8320.60
2,2015-01-13,-0.000821,8339.30,8303.20
3,2015-01-14,-0.000536,8302.80,8270.20
4,2015-01-15,-0.000771,8418.70,8508.25


# Volatility Measurement

Volatility is estimated using rolling return statistics and used to classify market conditions.

Each observation is assigned to a volatility regime for further analysis.

In [7]:
research_vol = research.merge(
    vol_df[
        ["date", "vol20"]
    ],
    on="date",
    how="left"
)

research_vol = research_vol.dropna()

research_vol.head()

,date,morning_return,entry_price,exit_price,vol20
20,2015-02-11,0.000482,8607.45,8629.60,0.009997
21,2015-02-12,-0.005768,8626.90,8718.60,0.010121
22,2015-02-13,-0.000074,8740.85,8807.40,0.010245
23,2015-02-16,0.001834,8847.60,8800.15,0.010263
24,2015-02-18,0.004052,8847.25,8867.25,0.008333


In [8]:
research_vol["vol_bucket"] = pd.qcut(
    research_vol["vol20"],
    3,
    labels=[
        "Low Vol",
        "Medium Vol",
        "High Vol"
    ]
)

research_vol.groupby(
    "vol_bucket"
)["vol20"].agg(
    ["min", "max", "count"]
)

,min,max,count
vol_bucket,,,
Low Vol,0.003254,0.006851,859
Medium Vol,0.006851,0.009354,858
High Vol,0.009354,0.056779,859




# Regime Comparison

Signal performance is evaluated separately across low, medium, and high volatility environments.

This analysis helps determine whether market conditions influence predictive power.

In [9]:
threshold = 0.0075

results = []

for bucket in ["Low Vol", "Medium Vol", "High Vol"]:

    subset = research_vol[
        research_vol["vol_bucket"] == bucket
    ]

    trade_returns = []

    for _, row in subset.iterrows():

        mr = row["morning_return"]

        # Long
        if mr > threshold:

            ret = (
                row["exit_price"]
                - row["entry_price"]
            ) / row["entry_price"]

        # Short
        elif mr < -threshold:

            ret = (
                row["entry_price"]
                - row["exit_price"]
            ) / row["entry_price"]

        else:
            continue

        ret -= 0.0005

        trade_returns.append(ret)

    trades = pd.Series(trade_returns)

    if len(trades) < 10:
        continue

    profit_factor = (
        trades[trades > 0].sum()
        /
        abs(trades[trades < 0].sum())
    )

    sharpe = (
        np.sqrt(len(trades))
        * trades.mean()
        / trades.std()
    )

    results.append([
        bucket,
        len(trades),
        trades.mean() * 100,
        sharpe,
        profit_factor
    ])

results_df = pd.DataFrame(
    results,
    columns=[
        "Volatility",
        "Trades",
        "Avg Trade %",
        "Sharpe",
        "Profit Factor"
    ]
)

print(results_df)

   Volatility  Trades  Avg Trade %    Sharpe  Profit Factor
0     Low Vol      36     0.016124  0.160277       1.069662
1  Medium Vol      67     0.116184  1.275819       1.503993
2    High Vol     128     0.205899  1.747105       1.634586


In [10]:
threshold = 0.0075

high_vol = research_vol[
    research_vol["vol_bucket"]  != "Low Vol"
]

trade_returns = []
trade_dates = []

for _, row in high_vol.iterrows():

    mr = row["morning_return"]

    # Long
    if mr > threshold:

        ret = (
            row["exit_price"]
            - row["entry_price"]
        ) / row["entry_price"]

    # Short
    elif mr < -threshold:

        ret = (
            row["entry_price"]
            - row["exit_price"]
        ) / row["entry_price"]

    else:
        continue

    ret -= 0.0005

    trade_returns.append(ret)
    trade_dates.append(row["date"])

trades = pd.Series(
    trade_returns,
    index=pd.to_datetime(trade_dates)
)

print("Trades:", len(trades))

Trades: 195


In [11]:
equity = (1 + trades).cumprod()

total_return = equity.iloc[-1] - 1

years = (
    (trades.index[-1]
     - trades.index[0]).days
    / 365.25
)

cagr = (
    equity.iloc[-1]
    ** (1 / years)
    - 1
)

max_dd = (
    equity / equity.cummax() - 1
).min()

profit_factor = (
    trades[trades > 0].sum()
    /
    abs(trades[trades < 0].sum())
)

win_rate = (
    trades > 0
).mean()

sharpe = (
    np.sqrt(len(trades) / years)
    * trades.mean()
    / trades.std()
)

print("=" * 60)
print("HIGH VOL MOMENTUM STRATEGY")
print("=" * 60)
print("Trades         :", len(trades))
print("Total Return   :", f"{total_return:.2%}")
print("CAGR           :", f"{cagr:.2%}")
print("Sharpe         :", round(sharpe, 2))
print("Max Drawdown   :", f"{max_dd:.2%}")
print("Win Rate       :", f"{win_rate:.2%}")
print("Profit Factor  :", round(profit_factor, 2))
print("Average Trade  :", f"{trades.mean():.2%}")
print("=" * 60)

HIGH VOL MOMENTUM STRATEGY
Trades         : 195
Total Return   : 38.84%
CAGR           : 3.26%
Sharpe         : 0.66
Max Drawdown   : -9.72%
Win Rate       : 55.90%
Profit Factor  : 1.6
Average Trade  : 0.18%


In [12]:
import plotly.express as px
import pandas as pd

regime_df = pd.DataFrame({
    "Regime": ["Low Vol", "Medium Vol", "High Vol"],
    "Profit Factor": [1.07, 1.50, 1.63],
    "Average Trade": [0.016, 0.116, 0.206]
})

fig = px.bar(
    regime_df,
    x="Regime",
    y="Profit Factor",
    hover_data=["Average Trade"],
    title="Profit Factor by Volatility Regime"
)

fig.show()

# Conclusions

## Research Question

Does signal performance depend on volatility regime?

## Evidence

- High-volatility periods generated superior returns.
- Profit factor improved in elevated volatility environments.
- Low-volatility periods produced weaker results.

## Verdict

🟢 Accepted

Volatility regime is an important determinant of signal performance.

High-volatility environments provide the strongest opportunities and should be considered when filtering trades.